# 01 Data Audit

Phase 1 focuses on analytical foundations: verify schema quality, inspect missingness, document cleaning logic, and confirm that normalized outputs preserve title coverage.

This notebook intentionally stays narrow. It is an audit and transformation notebook, not the main business analysis deliverable.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/netflix_titles.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.cleaning import build_processed_outputs, read_raw_titles

RAW_PATH = PROJECT_ROOT / "data/raw/netflix_titles.csv"
PROCESSED_PATH = PROJECT_ROOT / "data/processed"


In [2]:
raw_df = read_raw_titles(RAW_PATH)
print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
display(raw_df.head(3))

Raw shape: 6,234 rows x 12 columns


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
2,70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."


In [3]:
schema_audit = pd.DataFrame(
    {
        "column": raw_df.columns,
        "dtype": raw_df.dtypes.astype(str).values,
        "non_null_count": raw_df.notna().sum().values,
        "missing_count": raw_df.isna().sum().values,
        "missing_share": raw_df.isna().mean().round(4).values,
    }
).sort_values(["missing_count", "column"], ascending=[False, True]).reset_index(drop=True)

display(schema_audit)

display(raw_df["type"].value_counts(dropna=False).rename_axis("type").reset_index(name="title_count"))
display(raw_df["rating"].fillna("Missing").value_counts(dropna=False).head(12).rename_axis("rating").reset_index(name="title_count"))

,column,dtype,non_null_count,missing_count,missing_share
0,director,object,4265,1969,0.3158
1,cast,object,5664,570,0.0914
2,country,object,5758,476,0.0764
3,date_added,object,6223,11,0.0018
4,rating,object,6224,10,0.0016
5,description,object,6234,0,0.0000
6,duration,object,6234,0,0.0000
7,listed_in,object,6234,0,0.0000
8,release_year,int64,6234,0,0.0000
9,show_id,string,6234,0,0.0000


,type,title_count
0,Movie,4265
1,TV Show,1969


,rating,title_count
0,TV-MA,2027
1,TV-14,1698
2,TV-PG,701
3,R,508
4,PG-13,286
5,NR,218
6,PG,184
7,TV-Y7,169
8,TV-G,149
9,TV-Y,143


In [4]:
date_parse_check = pd.DataFrame(
    {
        "check": [
            "raw_missing_date_added",
            "naive_parse_failures",
            "explicit_parse_failures_after_trim",
        ],
        "value": [
            raw_df["date_added"].isna().sum(),
            pd.to_datetime(raw_df["date_added"], errors="coerce").isna().sum(),
            pd.to_datetime(
                raw_df["date_added"].astype("string").str.strip(),
                format="%B %d, %Y",
                errors="coerce",
            ).isna().sum(),
        ],
    }
)

display(date_parse_check)

,check,value
0,raw_missing_date_added,11
1,naive_parse_failures,651
2,explicit_parse_failures_after_trim,11


## Cleaning Rules Applied in Phase 1

- Preserve all raw title rows.
- Trim whitespace before parsing `date_added` with the explicit format `%B %d, %Y`.
- Keep missing values visible instead of dropping rows globally.
- Normalize `country`, `listed_in`, `cast`, and `director` into bridge tables.
- Preserve raw `rating` and add heuristic portfolio groupings with `rating_system`, `rating_group`, and `is_unrated`.
- Preserve raw `duration` and parse it into `duration_value` plus `duration_unit`.

In [5]:
processed_tables, qa_outputs = build_processed_outputs(raw_df)

display(qa_outputs["qa_table_counts"])
display(qa_outputs["qa_parse_checks"])
display(qa_outputs["qa_bridge_coverage"])
display(qa_outputs["qa_titles_missingness"].head(10))

,table_name,row_count,distinct_show_id
0,raw_titles,6234,6234
1,titles,6234,6234
2,title_country,7179,5758
3,title_genre,13670,6234
4,title_cast,44310,5664
5,title_director,4851,4265


,check_name,value
0,duplicate_show_id_raw,0
1,duplicate_show_id_titles,0
2,raw_date_added_missing,11
3,date_added_parse_failures_non_missing,0
4,duration_parse_failures,0
5,unrated_or_unknown_titles,235


,source_column,bridge_table,titles_with_values_raw,titles_with_values_bridge,bridge_rows,avg_values_per_title
0,country,title_country,5758,5758,7179,1.25
1,listed_in,title_genre,6234,6234,13670,2.19
2,cast,title_cast,5664,5664,44310,7.82
3,director,title_director,4265,4265,4851,1.14


,column,missing_count,missing_share
0,date_added,11,0.0018
1,date_added_month,11,0.0018
2,date_added_year,11,0.0018
3,rating,10,0.0016
4,description,0,0.0000
5,duration,0,0.0000
6,duration_unit,0,0.0000
7,duration_value,0,0.0000
8,is_unrated,0,0.0000
9,rating_group,0,0.0000


In [6]:
processed_inventory = pd.DataFrame(
    {"file_name": sorted(path.name for path in PROCESSED_PATH.glob("*.csv"))}
)
display(processed_inventory)

,file_name
0,qa_bridge_coverage.csv
1,qa_parse_checks.csv
2,qa_table_counts.csv
3,qa_titles_missingness.csv
4,title_cast.csv
5,title_country.csv
6,title_director.csv
7,title_genre.csv
8,titles.csv


## Phase 1 Outcome

At the end of this notebook, the project has an analysis-ready base table, normalized entity bridges, and QA artifacts that can support later business analysis modules without redoing cleaning logic.